# Car Price Prediction via Regularized Regression
## A Comparative Study of Ridge, Lasso, and ElasticNet Approaches
### Author: MBulah
---
**Dataset:** CarPrice_Assignment.csv &nbsp;|&nbsp; 205 vehicles × 26 attributes  
**Prediction Target:** Selling price in USD  
**GitHub:** https://github.com/bulah16/C4-Project.git

**Study Objective:** Geely Auto, a Chinese vehicle manufacturer, intends to establish a presence in the North American automobile market. This notebook constructs and evaluates supervised regression models — ordinary least-squares Linear Regression alongside three penalty-based variants (Ridge, Lasso, ElasticNet) — to identify which vehicle attributes drive pricing and which estimator delivers the most reliable predictions on held-out data.


## 1. Library Imports and Data Loading

All required packages are imported upfront and the dataset is read from CSV. A structural check verifies row/column counts and confirms zero missing values before analysis begins.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 110, 'figure.autolayout': True})

df = pd.read_csv('CarPrice_Assignment.csv')
print(f"Dimensions  : {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Missing cells: {df.isnull().sum().sum()}")
df.head()

## 2. Exploratory Data Analysis (EDA)

Exploratory analysis lays the groundwork for every subsequent modelling decision. The subsections below examine dataset structure, the target variable's distributional shape, categorical pricing signals, numerical correlations, and potential outliers.

### 2.1 Initial Inspection of the Dataset

Summary statistics and data-type information establish a baseline understanding of the feature space.

In [ ]:
print("── Data Types & Non-Null Counts ──")
df.info()
print("\n── Descriptive Statistics ──")
df.describe().round(2)

### 2.2 Distribution of the Target Variable (Price)

The price variable is examined for skewness. A log1p transformation is visualised alongside the raw distribution to assess whether a target transformation would benefit modelling.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['price'], bins=30, color='#3B82F6', edgecolor='white', linewidth=0.6)
axes[0].set_title('Figure 1a: Distribution of Car Prices', fontsize=11)
axes[0].set_xlabel('Price (USD)'); axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['price']), bins=30, color='#F97316', edgecolor='white', linewidth=0.6)
axes[1].set_title('Figure 1b: Log-Transformed Price Distribution', fontsize=11)
axes[1].set_xlabel('Log(Price + 1)'); axes[1].set_ylabel('Frequency')

plt.suptitle('Figure 1: Raw vs. Log-Transformed Price Distributions', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

stats = df['price'].agg(['min','mean','median','max','skew']).rename(
    {'min':'Minimum','mean':'Mean','median':'Median','max':'Maximum','skew':'Skewness'})
print(stats.round(2).to_string())

### 2.3 Extracting the Car Manufacturer from CarName

The `CarName` field combines make and model. The manufacturer is isolated by extracting and lowercasing the first token. Six misspelled entries are corrected via a lookup dictionary.

In [ ]:
df['carcompany'] = df['CarName'].str.split().str[0].str.lower()

corrections = {
    'maxda': 'mazda', 'toyouta': 'toyota',
    'vokswagen': 'volkswagen', 'vw': 'volkswagen',
    'porcshce': 'porsche', 'alfa-romero': 'alfa-romeo'
}
df['carcompany'] = df['carcompany'].replace(corrections)

print(f"Distinct manufacturers after cleaning: {df['carcompany'].nunique()}")
print("\nRecord count per manufacturer:")
print(df['carcompany'].value_counts().to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
df['carcompany'].value_counts().plot(kind='bar', ax=ax, color='#0F766E', edgecolor='white', linewidth=0.6)
ax.set_title('Figure 2: Number of Cars per Company', fontsize=12, fontweight='bold')
ax.set_xlabel('Car Company'); ax.set_ylabel('Count')
plt.xticks(rotation=40, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

### 2.4 Categorical Attributes and Their Relationship with Price

Mean sale price is computed per category for six categorical columns. Bar charts reveal which categories command market premiums.

In [ ]:
cat_cols = ['fueltype','aspiration','doornumber','carbody','drivewheel','enginelocation']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(cat_cols):
    means = df.groupby(col)['price'].mean().sort_values()
    means.plot(kind='bar', ax=axes[i], color='#3B82F6', edgecolor='white', linewidth=0.5)
    axes[i].set_title(f'Avg Price by {col}', fontsize=10)
    axes[i].set_ylabel('Average Price (USD)'); axes[i].set_xlabel(col)
    axes[i].tick_params(axis='x', rotation=25)
plt.suptitle('Figure 3: Average Car Price by Categorical Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.5 Correlation Structure of Numerical Attributes

Pairwise Pearson correlations are displayed as a lower-triangular heatmap. The six attributes with the highest absolute correlation with price are then examined via scatter plots.

In [ ]:
num_cols = ['symboling','wheelbase','carlength','carwidth','carheight',
            'curbweight','enginesize','boreratio','stroke','compressionratio',
            'horsepower','peakrpm','citympg','highwaympg','price']

corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.4, ax=ax, annot_kws={'size': 8})
ax.set_title('Figure 4: Correlation Heatmap of Numerical Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nTop correlations with price (ranked by absolute value):")
print(corr['price'].drop('price').sort_values(key=abs, ascending=False).round(3).to_string())

In [ ]:
top6 = corr['price'].drop('price').abs().sort_values(ascending=False).head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(top6):
    r = df[[col, 'price']].corr().iloc[0, 1]
    axes[i].scatter(df[col], df['price'], alpha=0.45, s=22, color='#1E3A5F')
    axes[i].set_xlabel(col); axes[i].set_ylabel('price')
    axes[i].set_title(f'{col} vs price  (r = {r:.2f})', fontsize=10)
plt.suptitle('Figure 5: Top Correlated Features vs Price', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

### 2.6 Outlier Screening via Box Plots

Box plots identify potential extreme values in the six most price-relevant attributes. Observations beyond 1.5×IQR are flagged but not removed — they correspond to verifiable high-specification vehicles.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for i, col in enumerate(top6):
    axes[i].boxplot(df[col].dropna(), patch_artist=True,
                    boxprops=dict(facecolor='#BAE6FD', color='#1E3A5F'),
                    medianprops=dict(color='#DC2626', linewidth=2),
                    flierprops=dict(marker='o', color='#F97316', markersize=5))
    axes[i].set_title(col, fontsize=10); axes[i].set_ylabel(col)
plt.suptitle('Figure 6: Boxplots of Key Numerical Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Three preparation steps convert the raw, partly categorical dataset into a model-ready feature matrix: categorical encoding, train/test partitioning, and feature standardization.

### 3.1 Categorical Variable Encoding

Binary attributes are mapped to 0/1 integers. Multi-level categoricals are one-hot encoded with `drop_first=True` to eliminate the redundant reference column.

In [ ]:
df_model = df.drop(columns=['car_ID', 'CarName'])

binary_map = {
    'gas': 1, 'diesel': 0, 'turbo': 1, 'std': 0,
    'two': 1, 'four': 0, 'front': 1, 'rear': 0
}
for col in ['fueltype', 'aspiration', 'doornumber', 'enginelocation']:
    df_model[col] = df_model[col].map(binary_map)

ohe_cols = ['carbody', 'drivewheel', 'enginetype', 'cylindernumber', 'fuelsystem', 'carcompany']
df_model = pd.get_dummies(df_model, columns=ohe_cols, drop_first=True)
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

print(f"Feature matrix dimensions after encoding: {df_model.shape}")
print(f"Missing values remaining       : {df_model.isnull().sum().sum()}")

### 3.2 Train / Test Partitioning and Feature Standardization

An 80/20 split (random_state=42) produces training and evaluation subsets. `StandardScaler` is calibrated on the training partition only and its stored parameters are applied to both subsets.

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training partition : {X_train.shape[0]} rows × {X_train.shape[1]} features")
print(f"Test partition     : {X_test.shape[0]}  rows × {X_test.shape[1]} features")

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print("\nStandardScaler applied — scaler was fitted on the training set only.")

## 4. Multicollinearity Assessment — Variance Inflation Factor (VIF)

VIF scores quantify the degree to which each numerical predictor is linearly explained by the others. Values above 10 signal problematic redundancy that can destabilise OLS coefficient estimates and motivate regularized alternatives.

In [ ]:
base_num = ['symboling','wheelbase','carlength','carwidth','carheight',
            'curbweight','enginesize','boreratio','stroke','compressionratio',
            'horsepower','peakrpm','citympg','highwaympg']

X_sc_df   = pd.DataFrame(X_train_sc, columns=X_train.columns)
vif_cols  = [c for c in X_train.columns if c in base_num]
X_vif_sub = X_sc_df[vif_cols]

vif_results = pd.DataFrame({
    'Feature': vif_cols,
    'VIF': [variance_inflation_factor(X_vif_sub.values, i) for i in range(len(vif_cols))]
}).sort_values('VIF', ascending=False).reset_index(drop=True)

print(vif_results.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#DC2626' if v > 10 else '#3B82F6' for v in vif_results['VIF']]
ax.barh(vif_results['Feature'], vif_results['VIF'], color=colors, edgecolor='white')
ax.axvline(10, color='#DC2626', linestyle='--', linewidth=1.5, label='VIF = 10 threshold')
ax.set_xlabel('Variance Inflation Factor (VIF)')
ax.set_title('Figure 10: VIF Scores for Numerical Attributes', fontsize=12, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

flagged = vif_results[vif_results['VIF'] > 10]['Feature'].tolist()
print(f"\nAttributes exceeding VIF = 10: {flagged}")

## 5. Regression Model Development

Four estimators are trained on the standardized training matrix. The three evaluation metrics reported throughout are RMSE (penalises large errors quadratically), MAE (average absolute error), and R² (proportion of variance explained).

### 5.1 Baseline — Ordinary Linear Regression (OLS)

A plain OLS model is fitted first to establish a reference benchmark. No penalty constrains coefficient magnitudes.

**Loss:** Minimise Σ (yᵢ − ŷᵢ)²

In [ ]:
lr = LinearRegression()
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
mae_lr  = mean_absolute_error(y_test, y_pred_lr)
r2_lr   = r2_score(y_test, y_pred_lr)

print("Ordinary Linear Regression (OLS) — Test Results")
print(f"  RMSE : ${rmse_lr:>10,.2f}")
print(f"  MAE  : ${mae_lr:>10,.2f}")
print(f"  R²   : {r2_lr:>11.4f}")

### 5.2 Ridge Regression with L2 Regularization

Ridge augments the OLS loss with an L2 penalty that shrinks all coefficients toward zero without eliminating any.

**Loss:** Minimise Σ (yᵢ − ŷᵢ)² + α · Σ βⱼ²

The regularization strength α is selected via 5-fold cross-validation.

In [ ]:
alphas   = [0.01, 0.1, 1, 10, 50, 100, 200, 500, 1000]
cv_r2    = [cross_val_score(Ridge(alpha=a), X_train_sc, y_train, cv=5, scoring='r2').mean()
            for a in alphas]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(alphas, cv_r2, marker='o', color='#0F766E', linewidth=2, markersize=7)
ax.set_xscale('log')
ax.set_xlabel('Alpha (log scale)'); ax.set_ylabel('Mean 5-Fold CV R²')
ax.set_title('Figure 12: Ridge — Alpha Tuning via 5-Fold Cross-Validation', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

best_alpha = alphas[int(np.argmax(cv_r2))]
print(f"Best alpha : {best_alpha}  (CV R² = {max(cv_r2):.4f})")

ridge = Ridge(alpha=best_alpha)
ridge.fit(X_train_sc, y_train)
y_pred_ridge = ridge.predict(X_test_sc)

print(f"\nRidge Regression (alpha={best_alpha}) — Test Results")
print(f"  RMSE : ${np.sqrt(mean_squared_error(y_test, y_pred_ridge)):>10,.2f}")
print(f"  MAE  : ${mean_absolute_error(y_test, y_pred_ridge):>10,.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_ridge):>11.4f}")

### 5.3 Lasso Regression with L1 Regularization

Lasso substitutes an absolute-value (L1) penalty. The L1 geometry can force individual coefficients to exactly zero, performing implicit variable selection.

**Loss:** Minimise Σ (yᵢ − ŷᵢ)² + α · Σ |βⱼ|

In [ ]:
lasso = Lasso(alpha=50, max_iter=10000)
lasso.fit(X_train_sc, y_train)
y_pred_lasso = lasso.predict(X_test_sc)

n_zero    = int(np.sum(lasso.coef_ == 0))
n_nonzero = int(np.sum(lasso.coef_ != 0))

print(f"Lasso Regression (alpha=50) — Test Results")
print(f"  RMSE     : ${np.sqrt(mean_squared_error(y_test, y_pred_lasso)):>10,.2f}")
print(f"  MAE      : ${mean_absolute_error(y_test, y_pred_lasso):>10,.2f}")
print(f"  R²       : {r2_score(y_test, y_pred_lasso):>11.4f}")
print(f"  Zero coefs    : {n_zero}  |  Non-zero coefs: {n_nonzero}")

In [ ]:
coef_df = (pd.DataFrame({'Feature': X_train.columns, 'Coefficient': lasso.coef_})
           .query('Coefficient != 0')
           .reindex(columns=['Feature','Coefficient'])
           .assign(AbsCoef=lambda d: d['Coefficient'].abs())
           .sort_values('AbsCoef', ascending=False)
           .head(20))

fig, ax = plt.subplots(figsize=(12, 5))
bar_colors = ['#1D4ED8' if v > 0 else '#DC2626' for v in coef_df['Coefficient']]
ax.bar(coef_df['Feature'], coef_df['Coefficient'], color=bar_colors, edgecolor='white', linewidth=0.5)
ax.set_title('Figure 11: Top 20 Non-Zero Lasso Coefficients (alpha=50)', fontsize=11, fontweight='bold')
ax.set_ylabel('Coefficient Value')
ax.axhline(0, color='black', linewidth=0.8)
ax.tick_params(axis='x', rotation=55)
plt.tight_layout()
plt.show()

### 5.4 ElasticNet — Blended L1 and L2 Regularization

ElasticNet merges both penalties. The mixing parameter `l1_ratio` (set to 0.5 here) controls the relative contribution of L1 versus L2.

**Loss:** Minimise Σ (yᵢ − ŷᵢ)² + α · [ρ · Σ|βⱼ| + (1−ρ) · Σβⱼ²]

In [ ]:
en = ElasticNet(alpha=50, l1_ratio=0.5, max_iter=10000)
en.fit(X_train_sc, y_train)
y_pred_en = en.predict(X_test_sc)

print(f"ElasticNet (alpha=50, l1_ratio=0.5) — Test Results")
print(f"  RMSE : ${np.sqrt(mean_squared_error(y_test, y_pred_en)):>10,.2f}")
print(f"  MAE  : ${mean_absolute_error(y_test, y_pred_en):>10,.2f}")
print(f"  R²   : {r2_score(y_test, y_pred_en):>11.4f}")

## 6. Cross-Model Performance Comparison

All four estimators are evaluated on the same held-out test partition. Actual-versus-predicted scatter plots and residual diagnostics accompany the summary metric table.

In [ ]:
def evaluate(y_true, y_pred, label):
    return {
        'Model': label,
        'RMSE ($)': round(np.sqrt(mean_squared_error(y_true, y_pred)), 2),
        'MAE ($)':  round(mean_absolute_error(y_true, y_pred), 2),
        'R²':       round(r2_score(y_true, y_pred), 4)
    }

comparison = pd.DataFrame([
    evaluate(y_test, y_pred_lr,    'Linear Regression (OLS)'),
    evaluate(y_test, y_pred_ridge, f'Ridge (alpha={best_alpha})'),
    evaluate(y_test, y_pred_lasso, 'Lasso (alpha=50)'),
    evaluate(y_test, y_pred_en,    'ElasticNet (alpha=50, l1_ratio=0.5)'),
])
print("=== Model Performance on Test Set (n=41) ===")
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
model_results = [
    (y_pred_lr,    'Linear Regression (OLS)'),
    (y_pred_ridge, f'Ridge (alpha={best_alpha})'),
    (y_pred_lasso, 'Lasso (alpha=50)'),
    (y_pred_en,    'ElasticNet'),
]
for ax, (preds, name) in zip(axes.flatten(), model_results):
    ax.scatter(y_test, preds, alpha=0.55, s=28, color='#1E3A5F')
    ceiling = max(float(y_test.max()), float(np.array(preds).max()))
    ax.plot([0, ceiling], [0, ceiling], 'r--', linewidth=1.5)
    ax.set_xlabel('Actual Price (USD)'); ax.set_ylabel('Predicted Price (USD)')
    ax.set_title(f'{name}\nR² = {r2_score(y_test, preds):.3f}', fontsize=10)
plt.suptitle('Figure 7: Actual vs. Predicted Prices — All Models', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, (preds, name) in zip(axes.flatten(), model_results):
    residuals = np.array(y_test) - np.array(preds)
    ax.scatter(preds, residuals, alpha=0.5, s=22, color='#0F766E')
    ax.axhline(0, color='#DC2626', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Predicted Price'); ax.set_ylabel('Residual (Actual − Predicted)')
    ax.set_title(name, fontsize=10)
plt.suptitle('Figure 8: Residual Plots — All Models', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
palette = ['#DC2626', '#3B82F6', '#16A34A']
for i, metric in enumerate(['RMSE ($)', 'MAE ($)', 'R²']):
    axes[i].bar(comparison['Model'], comparison[metric],
                color=palette[i], edgecolor='white', width=0.55)
    axes[i].set_title(f'{metric}', fontsize=11, fontweight='bold')
    axes[i].set_ylabel(metric)
    axes[i].tick_params(axis='x', rotation=18)
plt.suptitle('Figure 9: Side-by-Side Model Performance Comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Findings, Assumptions, and Strategic Recommendations

### 7.1 Principal Findings

1. **Price concentration and spread:** The sale price distribution is right-skewed (skewness ≈ 1.78). The median transaction sits near \$10,295 while the mean reaches \$13,277, indicating that a small cluster of premium vehicles pulls the average upward.

2. **Dominant positive pricing drivers:** Engine displacement (r = 0.874), vehicle weight (r = 0.835), rated power output (r = 0.808), and body width (r = 0.759) are the four strongest upward predictors — capturing the market premium placed on performance and size.

3. **Dominant negative pricing drivers:** Highway fuel economy (r = −0.698) and city fuel economy (r = −0.686) move inversely with price, positioning compact, efficient vehicles in lower price brackets.

4. **Manufacturer premium:** Brand affiliation contributes meaningfully to predicted price. Luxury marques — notably Jaguar, Buick, and Porsche — command average prices substantially above the dataset mean even after controlling for mechanical specifications.

5. **Multicollinearity:** City MPG (VIF = 25.86), highway MPG (VIF = 20.18), and curb weight (VIF = 15.46) each exceed VIF = 10, forming a cluster of inter-dependent predictors that destabilises plain OLS estimation.

6. **Best overall accuracy:** Ordinary Linear Regression achieved the strongest test-set result (R² = 0.9097, RMSE = \$2,669.93), attributable in part to the dataset being sufficiently compact that OLS does not catastrophically overfit despite 64 features.

7. **Lasso as an interpretability tool:** At α = 50, Lasso eliminated 25 of 64 predictors while maintaining R² = 0.8859, producing a sparse model that is easier to communicate to non-technical stakeholders.

8. **Ridge as a stability-preserving alternative:** Ridge (α = 10, R² = 0.8914) retains all predictors but constrains their magnitudes, providing smoother coefficient estimates in the presence of correlated inputs.

9. **ElasticNet with default blending is over-penalised:** The combined L1+L2 penalty at α = 50 and l1_ratio = 0.5 yielded R² = 0.3792 — a targeted joint grid search over both hyperparameters is needed before conclusions can be drawn.

---

### 7.2 Documented Modelling Assumptions

- No imputation was performed; the dataset is fully populated with zero missing values across all 26 columns.
- The car manufacturer was defined as the first whitespace-delimited token of `CarName`, lowercased, and corrected for six documented misspellings.
- Binary categorical attributes were label-encoded (0/1); multi-level categoricals were one-hot encoded with the reference category dropped.
- `StandardScaler` was calibrated exclusively on the training partition and its stored parameters applied to both subsets — preventing test-set leakage.
- All four models share the same 80/20 random split (random_state = 42) for directly comparable evaluation.
- Extreme observations in engine size and horsepower were verified as legitimate high-specification vehicles and retained.
- Ridge alpha was optimised via 5-fold CV on the training set only; Lasso and ElasticNet used a fixed alpha of 50 as a starting configuration.
- ElasticNet l1_ratio was fixed at 0.5; this parameter was not independently optimised.
- All monetary figures are in US dollars as supplied in the source file.

---

### 7.3 Strategic Recommendations for Geely Auto

- **Specification targeting:** Prioritise engine displacement, vehicle weight, output power, and body width as the primary engineering levers when positioning a new model within a target price bracket.
- **Target transformation:** A log1p transformation of the price target should be explored in subsequent iterations; the observed positive skew can produce heteroscedastic residuals that violate OLS assumptions.
- **Collinearity reduction:** The citympg / highwaympg / curbweight cluster warrants further investigation. Removing one or constructing a composite index would simplify the feature space without sacrificing much predictive power.
- **ElasticNet re-evaluation:** ElasticNet deserves a second assessment with a full grid search over both alpha and l1_ratio before being discounted.
- **Lasso feature subset:** The 39-feature Lasso subset provides a strong candidate shortlist for a leaner, more transparent production pricing model.
- **Brand strategy:** Lasso coefficients show that manufacturer dummies for premium marques carry some of the largest positive weights — brand equity is a measurable, actionable pricing lever.
